### Build a Simple Workflow or Graph Using LangGraph

#### State
First, define the **State** of the graph.

The **State schema** serves as the central data structure and input schema for all Nodes and Edges in the graph.

Let's use the `TypedDict` class from Python's standard `typing` module as our schema, which provides type hints for the keys.


In [ ]:
from typing import TypedDict

class State(TypedDict):
    graph_info: str


#### Nodes
Nodes are just standard Python functions.

- The first positional argument is the **state**, as defined above.
- Because the state is a `TypedDict` with the schema defined above, each node can access the key `graph_info` using `state['graph_info']`.
- Each node returns a dictionary containing new values for state keys (e.g. `{"graph_info": ...}`).
- By default, the new value returned by each node will overwrite the prior state value for that key.


In [ ]:
def start_play(state: State) -> dict:
    print("--- start_play node has been called ---")
    return {"graph_info": state['graph_info'] + " I am planning to play"}

def cricket(state: State) -> dict:
    print("--- cricket node has been called ---")
    return {"graph_info": state['graph_info'] + " Cricket"}

def badminton(state: State) -> dict:
    print("--- badminton node has been called ---")
    return {"graph_info": state['graph_info'] + " Badminton"}


In [ ]:
import random
from typing import Literal

def random_play(state: State) -> Literal['cricket', 'badminton']:
    graph_info = state['graph_info']

    if random.random() > 0.5:
        return "cricket"
    else:
        return "badminton" 


#### Graph Construction
Now, we build the graph from our components defined above:

1. **StateGraph**: The central graph builder class initialized with our `State` schema.
2. **Nodes**: Added with `builder.add_node(node_name, node_func)`.
3. **Edges**:
   - `START`: A special entry-point node that receives the initial user input and directs it into the graph.
   - `add_conditional_edges`: Dynamically routes execution from `start_play` to either `cricket` or `badminton` based on `random_play`.
   - `END`: A special terminal node representing graph completion.
4. **Compile**: Call `builder.compile()` to validate the graph structure and generate an executable `CompiledGraph`.
5. **Visualization**: We can visualize the graph structure as a Mermaid diagram.


In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

## Build Graph
builder = StateGraph(State)

## Adding the nodes
builder.add_node("start_play", start_play)
builder.add_node("cricket", cricket)
builder.add_node("badminton", badminton)

## Schedule the flow of the graph
builder.add_edge(START, "start_play")
builder.add_conditional_edges("start_play", random_play)
builder.add_edge("cricket", END)
builder.add_edge("badminton", END)

## Compile the graph
graph = builder.compile()

## View the graph
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback to mermaid markdown / ascii if png rendering is not available
    print(graph.get_graph().draw_mermaid())


### Graph Invocation
We can now invoke our compiled graph by passing an initial state dictionary conforming to `State`.


In [ ]:
result = graph.invoke({"graph_info": "Hey My name is Krish"})
print("\nFinal State Output:")
print(result)
